# 01. Data Collection

This notebook collects:
- **BTC/USDT price data** (1-min + 1-hour candles) via Binance REST API
- **News headlines** via CryptoPanic API and RSS feeds (CoinDesk, CoinTelegraph)
- **Market statistics** (Market Cap, ATH, Circulating Supply) via CoinGecko API

> Set your `CRYPTOPANIC_API_KEY` in a `.env` file in the project root.

In [7]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv()

from src.collector import (
    fetch_binance_ticker_24h,
    collect_binance_klines_range,
    fetch_cryptopanic_news,
    fetch_all_rss,
    fetch_coingecko_stats,
    load_all_news,
    load_klines,
)
from datetime import datetime, timezone
import pandas as pd

## 1. Binance 24h Ticker (current price)

In [8]:
ticker = fetch_binance_ticker_24h()
print(f"Current BTC Price : ${ticker['price']:,.2f}")
print(f"24h Change        : {ticker['change_pct_24h']:+.2f}%")
print(f"24h High / Low    : ${ticker['high_24h']:,.2f} / ${ticker['low_24h']:,.2f}")
ticker

Current BTC Price : $74,718.03
24h Change        : -0.90%
24h High / Low    : $76,240.66 / $73,724.31


{'symbol': 'BTCUSDT',
 'price': 74718.03,
 'change_pct_24h': -0.897,
 'high_24h': 76240.66,
 'low_24h': 73724.31,
 'volume_24h': 13263.67898,
 'fetched_at': '2026-04-20T06:53:52.041218+00:00'}

## 2. Historical Kline Collection

- `interval='1h'` for visualization background (fast, ~80k rows)
- `interval='1m'` for analysis (slow, ~4.7M rows — run overnight if needed)

Run the 1h collection first to verify connectivity.

In [ ]:
# 1-hour candles (for timeseries chart)
# collect_binance_klines_range(
#     symbol='BTCUSDT',
#     interval='1h',
#     start=datetime(2017, 1, 1, tzinfo=timezone.utc),
# )

2026-04-20 15:53:52,075 [INFO] Starting klines collection: BTCUSDT 1h  2017-01-01 → 2026-04-20
2026-04-20 15:53:52,097 [INFO] Resuming from 2026-04-20 05:00:00+00:00 (file: btc_1h_202604.parquet)
2026-04-20 15:53:52,453 [INFO] Done. Total rows: 3


In [32]:
# 1-minute candles (for correlation analysis) — long-running!
# Uncomment to start:
collect_binance_klines_range(
    symbol='BTCUSDT',
    interval='1m',
    start=datetime(2017, 1, 1, tzinfo=timezone.utc),
)

2026-04-20 17:17:44,063 [INFO] Starting klines collection: BTCUSDT 1m  2017-01-01 → 2026-04-20
2026-04-20 17:18:07,781 [INFO]   collected 100000 rows, current: 2017-10-25 19:59:00+00:00
2026-04-20 17:18:30,362 [INFO]   collected 200000 rows, current: 2018-01-03 07:04:00+00:00
2026-04-20 17:18:54,689 [INFO]   collected 300000 rows, current: 2018-03-15 04:29:00+00:00
2026-04-20 17:19:18,935 [INFO]   collected 400000 rows, current: 2018-05-23 13:29:00+00:00
2026-04-20 17:19:44,660 [INFO]   collected 500000 rows, current: 2018-08-01 17:51:00+00:00
2026-04-20 17:20:11,286 [INFO]   collected 600000 rows, current: 2018-10-10 02:51:00+00:00
2026-04-20 17:20:35,275 [INFO]   collected 700000 rows, current: 2018-12-18 22:21:00+00:00
2026-04-20 17:21:00,231 [INFO]   collected 800000 rows, current: 2019-02-26 07:21:00+00:00
2026-04-20 17:21:25,053 [INFO]   collected 900000 rows, current: 2019-05-06 22:21:00+00:00
2026-04-20 17:21:51,008 [INFO]   collected 1000000 rows, current: 2019-07-15 18:22:00+

## 3. Verify Collected Klines

In [11]:
klines_1h = load_klines('1h')
print(f"1h rows: {len(klines_1h):,}")
print(f"Date range: {klines_1h['open_time'].min()} → {klines_1h['open_time'].max()}")
klines_1h.tail(3)

2026-04-20 15:53:52,832 [INFO] Loaded 75907 kline rows (1h) from 105 files


1h rows: 75,907
Date range: 2017-08-17 04:00:00+00:00 → 2026-04-20 06:00:00+00:00


,open_time,open,high,low,close,volume,close_time
75904,2026-04-20 04:00:00+00:00,74634.01,74637.43,74435.19,74567.64,333.76794,2026-04-20 04:59:59.999000+00:00
75905,2026-04-20 05:00:00+00:00,74567.63,74567.63,74260.03,74269.78,190.47991,2026-04-20 05:59:59.999000+00:00
75906,2026-04-20 06:00:00+00:00,74264.16,74894.11,74250.00,74718.03,366.38242,2026-04-20 06:59:59.999000+00:00


## 4. CryptoPanic News Collection

In [12]:
CRYPTOPANIC_KEY = os.getenv('CRYPTOPANIC_API_KEY', '')
if not CRYPTOPANIC_KEY:
    print('⚠  CRYPTOPANIC_API_KEY not set in .env — skipping API fetch')
else:
    news_df = fetch_cryptopanic_news(CRYPTOPANIC_KEY, pages=10)
    print(f'Fetched {len(news_df)} news items')
    display(news_df.head(5))

⚠  CRYPTOPANIC_API_KEY not set in .env — skipping API fetch


## 5. RSS Feed Collection (CoinDesk + CoinTelegraph)

In [13]:
rss_df = fetch_all_rss()
print(f'RSS headlines fetched: {len(rss_df)}')
rss_df.head(5)

2026-04-20 15:53:53,075 [INFO] RSS [coindesk] fetched 0 items
2026-04-20 15:53:53,330 [INFO] Saved 30 news records
2026-04-20 15:53:53,332 [INFO] RSS [cointelegraph] fetched 30 items


RSS headlines fetched: 30


,title,url,source,published_at
0,The quantum gap: Why Bitcoin and Ethereum are ...,https://cointelegraph.com/explained/bitcoin-an...,cointelegraph,2026-04-20 05:57:49+00:00
1,Polymarket in talks to raise $400M at a $15B v...,https://cointelegraph.com/news/polymarket-seek...,cointelegraph,2026-04-20 05:08:17+00:00
2,Hackers impersonated eth.limo team to hijack i...,https://cointelegraph.com/news/eth-limo-domain...,cointelegraph,2026-04-20 04:57:35+00:00
3,Bitcoin erases weekend gains as US-Iran ceasef...,https://cointelegraph.com/news/bitcoin-erases-...,cointelegraph,2026-04-20 04:15:27+00:00
4,Aave's TVL tanks $8B a day after $293M Kelp DA...,https://cointelegraph.com/news/aave-tvl-falls-...,cointelegraph,2026-04-20 03:13:37+00:00


## 6. CoinGecko Market Statistics

In [14]:
market = fetch_coingecko_stats()
print(f"Market Cap  : ${market['market_cap_usd']/1e12:.2f}T")
print(f"ATH         : ${market['ath_usd']:,.0f}")
print(f"Circ Supply : {market['circulating_supply']:,.0f} BTC")
market

2026-04-20 15:53:53,703 [INFO] CoinGecko stats saved → C:\Users\조동희\Desktop\KOSTA\프로젝트\Crypto-Trend-Analysis\data\processed\market_stats.json


Market Cap  : $1.50T
ATH         : $126,080
Circ Supply : 20,018,418 BTC


{'market_cap_usd': 1496163097357,
 'ath_usd': 126080,
 'circulating_supply': 20018418.0,
 'total_supply': 20018418.0,
 'price_usd': 74725,
 'price_change_24h_pct': -0.91963,
 'fetched_at': '2026-04-20T06:53:53.702801+00:00'}

## 7. Load All News & Summary

In [15]:
all_news = load_all_news()
print(f'Total news records: {len(all_news):,}')
if not all_news.empty:
    print(f"Date range: {all_news['published_at'].min()} → {all_news['published_at'].max()}")
    print(f"Sources: {all_news['source'].value_counts().to_dict()}")
all_news.head(5)

2026-04-20 15:53:53,751 [INFO] Loaded 30 news records from 1 files


Total news records: 30
Date range: 2026-04-17 19:14:41+00:00 → 2026-04-20 05:57:49+00:00
Sources: {'cointelegraph': 30}


,title,url,source,published_at
0,Ether accumulation wallet balances increased b...,https://cointelegraph.com/news/ether-accumulat...,cointelegraph,2026-04-17 19:14:41+00:00
1,US Senator asks for Binance monitor update ami...,https://cointelegraph.com/news/us-senator-bina...,cointelegraph,2026-04-17 19:22:53+00:00
2,Polymarket odds of Hormuz Strait traffic norma...,https://cointelegraph.com/news/polymarket-stra...,cointelegraph,2026-04-17 21:12:19+00:00
3,Russia introduces bill to criminalize unregist...,https://cointelegraph.com/news/russia-criminal...,cointelegraph,2026-04-17 21:33:01+00:00
4,Worldcoin tanks 13% as World’s iris-scanning t...,https://cointelegraph.com/news/worldcoin-falls...,cointelegraph,2026-04-18 00:49:26+00:00


In [ ]:
type(all_news)

pandas.core.frame.DataFrame

In [19]:
all_news.__len__()

30

In [35]:
all_news['url'].tolist()[0]

'https://cointelegraph.com/news/ether-accumulation-wallet-balances-increased-by-33percent-is-a-rally-to-dollar3k-next?utm_source=rss_feed&utm_medium=rss&utm_campaign=rss_partner_inbound'

_비트코인 데이터_
> `.parquet (파케이)

In [33]:
parquet_file = '../data/raw/btc_1m_202604.parquet'
df_parquet = pd.read_parquet(parquet_file)

In [34]:
df_parquet

,open_time,open,high,low,close,volume,close_time
0,2026-04-01 00:00:00+00:00,68284.49,68284.49,68218.35,68218.35,14.75370,2026-04-01 00:00:59.999000+00:00
1,2026-04-01 00:01:00+00:00,68218.36,68218.36,68120.99,68164.79,18.83543,2026-04-01 00:01:59.999000+00:00
2,2026-04-01 00:02:00+00:00,68164.79,68177.42,68142.66,68177.41,6.56973,2026-04-01 00:02:59.999000+00:00
3,2026-04-01 00:03:00+00:00,68177.40,68227.01,68177.40,68196.00,8.69624,2026-04-01 00:03:59.999000+00:00
4,2026-04-01 00:04:00+00:00,68196.01,68203.20,68190.52,68197.33,4.17840,2026-04-01 00:04:59.999000+00:00
...,...,...,...,...,...,...,...
27853,2026-04-20 08:13:00+00:00,74790.96,74840.82,74790.21,74840.82,13.51951,2026-04-20 08:13:59.999000+00:00
27854,2026-04-20 08:14:00+00:00,74840.81,74876.01,74811.54,74833.60,5.33837,2026-04-20 08:14:59.999000+00:00
27855,2026-04-20 08:15:00+00:00,74833.60,74887.99,74824.00,74852.52,3.66443,2026-04-20 08:15:59.999000+00:00
27856,2026-04-20 08:16:00+00:00,74852.52,74875.61,74845.51,74854.84,2.01385,2026-04-20 08:16:59.999000+00:00
